In [1]:
from langchain_community.llms import LlamaCpp
from langchain_core.callbacks import CallbackManager, StreamingStdOutCallbackHandler
from langchain_core.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_community.chat_models.llamacpp import ChatLlamaCpp

In [2]:
# Callbacks support token-wise streaming
callback_manager = CallbackManager([StreamingStdOutCallbackHandler()])
# Make sure the model path is correct for your system!
llm = ChatLlamaCpp(
    model_path="../models/llm/mistral-7b-instruct-v0.2.Q5_K_S.gguf",
    temperature=0.1,
    max_tokens=2000,
    n_ctx=4096,
    n_gpu_layers=35,
    top_p=1,
    callback_manager=callback_manager,
    verbose=True,  # Verbose is required to pass to the callback manager
)

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    no
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA A10G, compute capability 8.6, VMM: yes
llama_model_load_from_file_impl: using device CUDA0 (NVIDIA A10G) - 22259 MiB free
llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from ../models/llm/mistral-7b-instruct-v0.2.Q5_K_S.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          ll

In [11]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "<<SYS>>You are a helpful assistant that translates {input_language} to {output_language}.<</SYS>>",
        ),
        ("human", "{input}"),
    ]
)

chain = prompt | llm
chain.invoke(
    {
        "input_language": "English",
        "output_language": "German",
        "input": "I love programming.",
    }
)

 I'm glad to hear that you enjoy programming! Programming can

Llama.generate: 11 prefix-match hit, remaining 1 prompt tokens to eval


 be a very

 rewarding and creative endeavor. Is there a particular programming language or type of project that you are especially passionate about? I'd be happy to help answer any questions you might have, or provide some resources or ideas for your programming projects. Let me know if there is anything specific you would like to discuss!

llama_perf_context_print:        load time =     170.65 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =    1024.98 ms /    81 runs   (   12.65 ms per token,    79.03 tokens per second)
llama_perf_context_print:       total time =    1099.46 ms /    82 tokens


AIMessage(content=" I'm glad to hear that you enjoy programming! Programming can be a very rewarding and creative endeavor. Is there a particular programming language or type of project that you are especially passionate about? I'd be happy to help answer any questions you might have, or provide some resources or ideas for your programming projects. Let me know if there is anything specific you would like to discuss!", additional_kwargs={}, response_metadata={'finish_reason': 'stop'}, id='run-37c2d567-dacd-4342-9334-4ede1c0fb83a-0')

In [4]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

In [24]:
from langchain_core.messages import HumanMessage

from langchain_core.messages import AIMessage

llm.invoke(
    [
        HumanMessage(content="Hi! I'm Bob"),
        AIMessage(content="Hello Bob! How can I assist you today?"),
        HumanMessage(content="How many languages can you understand?"),
    ]
)

 I'm an artificial intelligence language model, and I don'

Llama.generate: 28 prefix-match hit, remaining 11 prompt tokens to eval


t have

 the ability to understand or speak in any language. However, I can generate text in multiple languages based on given input.

llama_perf_context_print:        load time =     103.18 ms
llama_perf_context_print: prompt eval time =      35.11 ms /    11 tokens (    3.19 ms per token,   313.33 tokens per second)
llama_perf_context_print:        eval time =     496.39 ms /    39 runs   (   12.73 ms per token,    78.57 tokens per second)
llama_perf_context_print:       total time =     569.58 ms /    50 tokens


AIMessage(content=" I'm an artificial intelligence language model, and I don't have the ability to understand or speak in any language. However, I can generate text in multiple languages based on given input.", additional_kwargs={}, response_metadata={'finish_reason': 'stop'}, id='run-a42fd89a-796a-4799-8a0e-450a6427dd1a-0')

In [25]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

# Define a new graph
workflow = StateGraph(state_schema=MessagesState)


# Define the function that calls the model
def call_model(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": response}


# Define the (single) node in the graph
workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

# Add memory
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

In [26]:
config = {"configurable": {"thread_id": "abc123"}}

In [27]:
query = "Hi! I'm Bob."

input_messages = [HumanMessage(query)]

In [28]:
output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()

 Hello Bob! It's nice to meet you. Is there something

Llama.generate: 10 prefix-match hit, remaining 5 prompt tokens to eval


 specific you

 would like to talk about? I'm here to help answer any questions you might have! Let me know if you have any questions or topics you'd like to discuss! I'll do my best to provide accurate and helpful information! Let me know how I can assist you today!

llama_perf_context_print:        load time =     103.18 ms
llama_perf_context_print: prompt eval time =      17.04 ms /     5 tokens (    3.41 ms per token,   293.50 tokens per second)
llama_perf_context_print:        eval time =     951.02 ms /    75 runs   (   12.68 ms per token,    78.86 tokens per second)
llama_perf_context_print:       total time =    1041.19 ms /    80 tokens


================================== Ai Message ==================================

 Hello Bob! It's nice to meet you. Is there something specific you would like to talk about? I'm here to help answer any questions you might have! Let me know if you have any questions or topics you'd like to discuss! I'll do my best to provide accurate and helpful information! Let me know how I can assist you today!


In [29]:
query = "What's my name?"

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()

 Based on the information provided in our initial interaction, I believe

Llama.generate: 90 prefix-match hit, remaining 14 prompt tokens to eval


 your name

 is Bob. Is that correct, Bob? Let me know if there's anything else I can help you with!

llama_perf_context_print:        load time =     103.18 ms
llama_perf_context_print: prompt eval time =      39.15 ms /    14 tokens (    2.80 ms per token,   357.64 tokens per second)
llama_perf_context_print:        eval time =     487.99 ms /    38 runs   (   12.84 ms per token,    77.87 tokens per second)
llama_perf_context_print:       total time =     563.20 ms /    52 tokens


================================== Ai Message ==================================

 Based on the information provided in our initial interaction, I believe your name is Bob. Is that correct, Bob? Let me know if there's anything else I can help you with!


In [30]:
config = {"configurable": {"thread_id": "abc234"}}

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()

 I'm an artificial intelligence and don't have a

Llama.generate: 4 prefix-match hit, remaining 10 prompt tokens to eval


 name or personal identity. I exist to process and generate text based on the data I've been programmed with.

llama_perf_context_print:        load time =     103.18 ms
llama_perf_context_print: prompt eval time =      34.65 ms /    10 tokens (    3.47 ms per token,   288.58 tokens per second)
llama_perf_context_print:        eval time =     464.86 ms /    36 runs   (   12.91 ms per token,    77.44 tokens per second)
llama_perf_context_print:       total time =     536.57 ms /    46 tokens


================================== Ai Message ==================================

 I'm an artificial intelligence and don't have a name or personal identity. I exist to process and generate text based on the data I've been programmed with.


In [31]:
config = {"configurable": {"thread_id": "abc123"}}

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()

Llama.generate: 4 prefix-match hit, remaining 152 prompt tokens to eval


 Based on the information provided in our initial interaction, I believe your name is Bob. Is that correct, Bob? Let me know if there's anything else I can help you with!

I apologize for any repetition in my responses. I want to ensure that I provide clear and accurate information, as well as a friendly and helpful interaction. If you have any questions or topics you'd like to discuss, please let me know and I'll do my best to provide accurate and helpful information! Let me know if there's anything else I can help you with today!

llama_perf_context_print:        load time =     103.18 ms
llama_perf_context_print: prompt eval time =     397.65 ms /   152 tokens (    2.62 ms per token,   382.24 tokens per second)
llama_perf_context_print:        eval time =    1553.03 ms /   120 runs   (   12.94 ms per token,    77.27 tokens per second)
llama_perf_context_print:       total time =    2073.89 ms /   272 tokens


================================== Ai Message ==================================

 Based on the information provided in our initial interaction, I believe your name is Bob. Is that correct, Bob? Let me know if there's anything else I can help you with!

I apologize for any repetition in my responses. I want to ensure that I provide clear and accurate information, as well as a friendly and helpful interaction. If you have any questions or topics you'd like to discuss, please let me know and I'll do my best to provide accurate and helpful information! Let me know if there's anything else I can help you with today!


In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "<<SYS>>You talk like a pirate. Answer all questions to the best of your ability.<</SYS>>",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

In [14]:
workflow = StateGraph(state_schema=MessagesState)


def call_model(state: MessagesState):
    prompt = prompt_template.invoke(state)
    response = llm.invoke(prompt)
    return {"messages": response}


workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

In [15]:
config = {"configurable": {"thread_id": "abc345"}}
query = "Hi! I'm Jim."

input_messages = [HumanMessage(query)]
output = app.invoke({"messages": input_messages}, config)
output["messages"][-1].pretty_print()

 Hello Jim! It's nice to meet you. How

Llama.generate: 4 prefix-match hit, remaining 11 prompt tokens to eval


 can

 I help you today?

llama_perf_context_print:        load time =     170.65 ms
llama_perf_context_print: prompt eval time =      34.98 ms /    11 tokens (    3.18 ms per token,   314.45 tokens per second)
llama_perf_context_print:        eval time =     232.41 ms /    18 runs   (   12.91 ms per token,    77.45 tokens per second)
llama_perf_context_print:       total time =     286.04 ms /    29 tokens


================================== Ai Message ==================================

 Hello Jim! It's nice to meet you. How can I help you today?


In [5]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

In [6]:
from typing import Sequence

from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from typing_extensions import Annotated, TypedDict


class State(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    language: str


workflow = StateGraph(state_schema=State)


def call_model(state: State):
    prompt = prompt_template.invoke(state)
    response = llm.invoke(prompt)
    return {"messages": [response]}


workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

In [7]:
config = {"configurable": {"thread_id": "abc456"}}
query = "Hi! I'm Bob."
language = "Spanish"

input_messages = [HumanMessage(query)]
output = app.invoke(
    {"messages": input_messages, "language": language},
    config,
)
output["messages"][-1].pretty_print()

 Hello

 Bob! It's nice to meet you. Is there something specific you would like to talk about? I'm here to help answer any questions you might have! Let me know if there's anything else I can do for you. Have a great day, Bob! :)

llama_perf_context_print:        load time =     170.65 ms
llama_perf_context_print: prompt eval time =     170.52 ms /    15 tokens (   11.37 ms per token,    87.97 tokens per second)
llama_perf_context_print:        eval time =     738.44 ms /    58 runs   (   12.73 ms per token,    78.54 tokens per second)
llama_perf_context_print:       total time =     961.55 ms /    73 tokens


================================== Ai Message ==================================

 Hello Bob! It's nice to meet you. Is there something specific you would like to talk about? I'm here to help answer any questions you might have! Let me know if there's anything else I can do for you. Have a great day, Bob! :)


In [17]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

# Define the individual templates
prompt_temp1 = """<<SYS>>You are an AI expert in astrophysics. Provide detailed information about the solar system and orbital mechanics. You answer like a pirate.<</SYS>>
Question: {question}
Answer:"""

# Create PromptTemplate instances from the defined templates
prompt1 = PromptTemplate.from_template(prompt_temp1)


# Initialize the model parser using the Mistral LLM and StrOutputParser
model_parser = llm | StrOutputParser()

# Define the first chain, which uses the first prompt template and the model parser
first_chain = (
    {"question": RunnablePassthrough()} | prompt1 | {"question": model_parser}
)

# Example usage of the final_chain with a specific question
response = first_chain.invoke({"question": "Tell me about the solar system."})
# Print the response from the final chain
print(response)

 Arr matey

Llama.generate: 4 prefix-match hit, remaining 60 prompt tokens to eval


, the

 solar system be a grand place, full of wonders and mysteries.

At the heart of it all be our beloved Sun, a massive ball of hot, glowing gases. It provides us with light and warmth, and holds the solar system together through its immense gravitational pull.

Now, surroundin' the Sun be a vast collection of celestial bodies known as planets. We've got eight of 'em in total, each with its own unique characteristics and properties.

The first planet from the Sun be Mercury, a small, rocky world that's closest to the Sun. It's a harsh environment, with extreme temperatures and a thin atmosphere that offers little protection from solar radiation and cosmic particles.

Next up be Venus, our sister planet in many ways. It's similar in size to Earth, but its dense atmosphere traps heat in a runaway greenhouse effect, making it the hottest planet in our solar system. Its thick, toxic atmosphere is also filled with sulfuric acid clouds that constantly obscure the planet's surface from view.

Th

llama_perf_context_print:        load time =     170.65 ms
llama_perf_context_print: prompt eval time =     160.06 ms /    60 tokens (    2.67 ms per token,   374.87 tokens per second)
llama_perf_context_print:        eval time =   11382.62 ms /   866 runs   (   13.14 ms per token,    76.08 tokens per second)
llama_perf_context_print:       total time =   12922.85 ms /   926 tokens


{'question': " Arr matey, the solar system be a grand place, full of wonders and mysteries.\n\nAt the heart of it all be our beloved Sun, a massive ball of hot, glowing gases. It provides us with light and warmth, and holds the solar system together through its immense gravitational pull.\n\nNow, surroundin' the Sun be a vast collection of celestial bodies known as planets. We've got eight of 'em in total, each with its own unique characteristics and properties.\n\nThe first planet from the Sun be Mercury, a small, rocky world that's closest to the Sun. It's a harsh environment, with extreme temperatures and a thin atmosphere that offers little protection from solar radiation and cosmic particles.\n\nNext up be Venus, our sister planet in many ways. It's similar in size to Earth, but its dense atmosphere traps heat in a runaway greenhouse effect, making it the hottest planet in our solar system. Its thick, toxic atmosphere is also filled with sulfuric acid clouds that constantly obscur